# MDDCC - Huan luyen tren Kaggle CPU

Tai hien Wang K. et al., *Scientific Reports* 14:16421 (2024),
DOI 10.1038/s41598-024-66907-z.

Notebook nay **idempotent**: lan chay thu N chi tiep tuc tu checkpoint tren S3,
khong tao `run_id` moi (tru khi xoa `current_run_id.json`).

Repo: https://github.com/richardnguyen1991/MDDCC.git

**Truoc khi chay, phai them 5 secret tren Kaggle** (Add-ons -> Secrets):
`AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`,
`S3_BUCKET`, `S3_PREFIX`. GitHub Secrets **khong** tu co mat trong runtime cua
Kaggle.

## 1. Fail-fast: dataset phai duoc mount

In [ ]:
# Muc 8.C: neu kernel-metadata.json thieu "dataset_sources" thi session do
# GitHub Actions khoi dong se KHONG co dataset, notebook chet ngay o buoc doc du
# lieu va vong lap restart quay vo ich. Kiem tra ngay dong dau.
import os
import sys
from pathlib import Path

ROOT = Path("/kaggle/input")
print("Co trong /kaggle/input:", sorted(p.name for p in ROOT.iterdir())
      if ROOT.exists() else "KHONG TON TAI")

parquets = sorted(ROOT.glob("**/*.parquet")) if ROOT.exists() else []
if not parquets:
    raise SystemExit(
        "FAIL-FAST: khong thay file .parquet nao duoi /kaggle/input.\n"
        "  - Neu chay tay: Add Input -> Datasets -> cicddos2019-parquet\n"
        "  - Neu do GitHub Actions push: kiem tra kernel/kernel-metadata.json\n"
        "    co khai bao \"dataset_sources\" khong (muc 8.C)."
    )

# Thu muc goc chung cua cac file parquet
DATA_DIR = parquets[0].parent
while not all(str(f).startswith(str(DATA_DIR) + os.sep) for f in parquets):
    DATA_DIR = DATA_DIR.parent
print(f"\nDATA_DIR = {DATA_DIR}")
print(f"So file parquet = {len(parquets)}")

## 2. Lay ma nguon

In [ ]:
# Roi khoi thu muc truoc khi xoa - neu dang dung trong do thi shell mat CWD
os.chdir("/kaggle/working")
WORK = "/kaggle/working/mddcc"

if Path(WORK).exists():
    import shutil
    shutil.rmtree(WORK, ignore_errors=True)

rc = os.system(f"git clone -q --depth 1 https://github.com/richardnguyen1991/MDDCC.git {WORK}")
if rc != 0 or not Path(WORK, "src", "train.py").exists():
    raise SystemExit(f"FAIL-FAST: git clone that bai (rc={rc}). Kiem tra "
                     "Settings -> Internet da bat chua.")
os.chdir(WORK)
sys.path.insert(0, WORK)
print("Commit:", os.popen("git rev-parse --short HEAD").read().strip())

## 3. Cai phu thuoc

In [ ]:
# Kaggle da co torch/numpy/sklearn/pyarrow. Chi cai nhung goi con thieu,
# phien ban ghim trong requirements.txt (muc 8.D).
os.system("pip install -q -r requirements.txt")

import numpy, sklearn, pyarrow, pywt, torch
print("torch  ", torch.__version__)
print("numpy  ", numpy.__version__)
print("sklearn", sklearn.__version__)
print("pyarrow", pyarrow.__version__)
print("pywt   ", pywt.__version__)
try:
    import shap
    print("shap   ", shap.__version__)
except ImportError:
    print("shap    KHONG CO -> se bo qua hinh C13")

## 4. Doc secret tu Kaggle

Credential AWS **khong** duoc nhet vao notebook, `kernel-metadata.json` hay git
(muc 8.A). Doc qua `kaggle_secrets`, fallback sang bien moi truong khi chay local.

In [ ]:
REQUIRED = ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY",
            "AWS_DEFAULT_REGION", "S3_BUCKET", "S3_PREFIX"]

try:
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()

    def read(name):
        try:
            return client.get_secret(name)
        except Exception:
            return os.environ.get(name, "")
except ImportError:
    print("Khong o Kaggle -> doc tu bien moi truong")

    def read(name):
        return os.environ.get(name, "")

missing = []
for name in REQUIRED:
    value = (read(name) or "").strip()
    if value:
        os.environ[name] = value
    else:
        missing.append(name)

if missing:
    raise SystemExit(
        "FAIL-FAST: thieu secret " + ", ".join(missing) + ".\n"
        "  Kaggle -> Add-ons -> Secrets, them dung 5 secret roi bat cho notebook nay.\n"
        "  GitHub Secrets KHONG tu co mat trong runtime cua Kaggle (muc 8.A)."
    )

# Chi in do dai, TUYET DOI khong in gia tri
for name in REQUIRED:
    v = os.environ[name]
    shown = v if name in ("AWS_DEFAULT_REGION", "S3_BUCKET", "S3_PREFIX") else f"<{len(v)} ky tu>"
    print(f"  {name:<24} {shown}")

## 5. Trang thai TRUOC khi chay

In [ ]:
import json
from src.s3io import store_from_env
from src.checkpoint import RunRegistry
import yaml

CFG_PATH = "configs/mddcc.yaml"
cfg = yaml.safe_load(Path(CFG_PATH).read_text(encoding="utf-8"))

# Duong dan mount that co the khac config (Kaggle dat ten theo slug) -> ghi de
cfg_dir = Path(cfg["data"]["kaggle_input_dir"])
if not cfg_dir.exists():
    print(f"CHU Y: {cfg_dir} khong ton tai, dung DATA_DIR do duoc: {DATA_DIR}")
    cfg["data"]["kaggle_input_dir"] = str(DATA_DIR)
    Path("configs/mddcc.runtime.yaml").write_text(
        yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    CFG_PATH = "configs/mddcc.runtime.yaml"

store = store_from_env(cfg)
run_id = RunRegistry(store).get()
print(f"run_id truoc khi chay = {run_id}")
if run_id:
    st = store.get_json_or_none(f"{run_id}/checkpoints/training_state.json") or {}
    print(f"  current_epoch = {st.get('current_epoch')} / {st.get('total_epochs')}")
    print(f"  status        = {st.get('status')}  exit_reason={st.get('exit_reason')}")
    print(f"  restart_count = {st.get('restart_count')}")
    if st.get("is_complete"):
        print("\n=> DA XONG 100 EPOCH. Session nay se chuyen sang buoc danh gia cuoi.")
else:
    print("  chua co run nao -> se tao run_id moi")

## 6. Huan luyen

Tu nap checkpoint + training state tu S3, tu luu dinh ky len S3, va tu thoat
truoc khi Kaggle cat session (`time_guard`). Thoat code 0 la binh thuong -
GitHub Actions se khoi dong session tiep theo.

In [ ]:
rc = os.system(f"python -m src.train --config {CFG_PATH} 2>&1")
print(f"\ntrain exit code = {rc}")
if rc != 0:
    raise SystemExit(f"src.train that bai (rc={rc}) - xem log phia tren")

## 7. Danh gia cuoi (chi khi da du 100 epoch)

In [ ]:
# Muc 4.8 + 7.E5: buoc RIENG BIET. Neu buoc nay loi thi checkpoint van nguyen
# tren S3 va chay lai duoc bang make_report.py.
run_id = RunRegistry(store).get()
st = store.get_json_or_none(f"{run_id}/checkpoints/training_state.json") or {}

if st.get("is_complete"):
    print("Du 100 epoch -> chay danh gia cuoi + sinh bao cao")
    rc_eval = os.system(f"python -m src.evaluate --config {CFG_PATH} 2>&1")
    print(f"evaluate exit code = {rc_eval}")
    if rc_eval == 0:
        bucket, prefix = os.environ["S3_BUCKET"], os.environ["S3_PREFIX"]
        rc_rep = os.system(
            f"python make_report.py --run-dir s3://{bucket}/{prefix}/{run_id} "
            f"--out /kaggle/working/report --upload 2>&1")
        print(f"make_report exit code = {rc_rep}")
else:
    print(f"Moi den epoch {st.get('current_epoch')}/{st.get('total_epochs')} "
          "-> chua danh gia. GitHub Actions se khoi dong session tiep theo.")

## 8. Trang thai SAU khi chay

In [ ]:
run_id = RunRegistry(store).get()
st = store.get_json_or_none(f"{run_id}/checkpoints/training_state.json") or {}
hist = store.get_json_or_none(f"{run_id}/metrics/history.json") or []

print("=" * 62)
print(f"run_id        = {run_id}")
print(f"session_id    = {st.get('session_id')}")
print(f"current_epoch = {st.get('current_epoch')} / {st.get('total_epochs')}")
print(f"status        = {st.get('status')}")
print(f"exit_reason   = {st.get('exit_reason')}")
print(f"restart_count = {st.get('restart_count')}")
print(f"so epoch trong history = {len(hist)}")
if hist:
    last = hist[-1]
    print(f"epoch cuoi    = {last['epoch']}  "
          f"val_macro_f1={last.get('val_macro_f1')}  "
          f"{last.get('epoch_seconds')}s")
    sessions = len({r.get("session_id") for r in hist})
    done = st.get("current_epoch") or 1
    remain = (st.get("total_epochs", 100) - done)
    avg = sum(r.get("epoch_seconds", 0) for r in hist) / max(len(hist), 1)
    print(f"so session da dung = {sessions}")
    print(f"uoc tinh con lai   = {remain} epoch x {avg:.0f}s = "
          f"{remain * avg / 3600:.1f}h")
print("=" * 62)